# 05 — Descriptors & Metaclasses
**References:** Ramalho *Fluent Python* Ch. 22–24 · Python docs: Descriptor HowTo Guide

## Narrative thread
```
Attribute lookup order -> Descriptor protocol -> Data vs non-data -> Metaclass machinery -> __new__ vs __init__
```

## 1. Attribute lookup order (MRO + descriptor protocol)

When you access `obj.attr`, Python follows this order:

1. **Data descriptors** in `type(obj).__mro__` (those with `__set__` or `__delete__`)
2. **Instance `__dict__`**
3. **Non-data descriptors** in `type(obj).__mro__` (only `__get__`)

A **descriptor** is any object whose class defines `__get__`, `__set__`, or `__delete__`.

- **Data descriptor:** `__get__` + (`__set__` or `__delete__`) — overrides instance dict
- **Non-data descriptor:** only `__get__` — overridden by instance dict

Functions are non-data descriptors: their `__get__` returns a bound method.

In [ ]:
import weakref
from typing import Any

# ── Data descriptor: validated attribute ──────────────────────────────────
class Validated:
    """Generic validated attribute descriptor."""
    def __set_name__(self, owner, name):   # called at class creation (3.6+)
        self.public_name  = name
        self.private_name = '_' + name     # store in instance dict under _name

    def __get__(self, obj, objtype=None):
        if obj is None:                    # accessed on class, not instance
            return self
        return getattr(obj, self.private_name, None)

    def __set__(self, obj, value):
        value = self.validate(value)       # subclass hook
        setattr(obj, self.private_name, value)

    def validate(self, value):             # override in subclasses
        return value


class PositiveFloat(Validated):
    def validate(self, value):
        value = float(value)
        if value <= 0:
            raise ValueError(f'{self.public_name} must be positive, got {value}')
        return value

class NonEmptyStr(Validated):
    def validate(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f'{self.public_name} must be a non-empty string')
        return value.strip()


class Product:
    name   = NonEmptyStr()     # __set_name__ sets public_name='name', private_name='_name'
    price  = PositiveFloat()
    weight = PositiveFloat()

    def __init__(self, name, price, weight):
        self.name   = name
        self.price  = price
        self.weight = weight

    def __repr__(self):
        return f'Product({self.name!r}, price={self.price}, weight={self.weight})'


p = Product('Widget', 9.99, 0.5)
print(p)

try:
    Product('', 9.99, 0.5)
except ValueError as e:
    print(f'Caught: {e}')

try:
    Product('Gadget', -5.0, 0.5)
except ValueError as e:
    print(f'Caught: {e}')

print(f'\nInstance __dict__: {p.__dict__}')  # note: _name, _price, _weight
print(f'Class descriptor:  {Product.__dict__["price"]}')  # the PositiveFloat instance

In [ ]:
# ── Non-data descriptor: how functions become methods ──────────────────────
class BoundMethodDemo:
    """Illustrate why functions accessed via instances return bound methods."""
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        # Equivalent to what function.__get__ does
        import functools
        return functools.partial(self.__call__, obj)

    def __call__(self, obj, *args, **kwargs):
        return f'called on {obj!r} with {args}, {kwargs}'


# Functions are non-data descriptors
def greet(self, name='World'):
    return f'Hello, {name}! I am {self.__class__.__name__}'

class MyClass:
    pass

MyClass.greet = greet  # add method post-hoc — same as defining in body
obj = MyClass()

print(f'Unbound:  {greet}  (plain function)')
print(f'Bound:    {obj.greet}  (bound method via __get__)')
print(f'Call:     {obj.greet("Python")}')  # self is injected automatically

## 2. Metaclasses

A **metaclass** is the class of a class. Just as `int` is the class of integers, `type` is the class of classes.

```
type(42)      → int
type(int)     → type
type(type)    → type        (type is its own metaclass)
```

**Class creation flow:**
1. Python evaluates the class body in a fresh namespace
2. Metaclass `__prepare__` provides the namespace dict (default: `dict`)
3. Metaclass `__new__` creates the class object
4. Metaclass `__init__` initializes it
5. `__set_name__` is called on all descriptors

In [ ]:
# ── Metaclass: auto-register subclasses ───────────────────────────────────
class PluginMeta(type):
    """Metaclass that registers every non-abstract subclass."""
    _registry: dict[str, type] = {}

    def __new__(mcs, name, bases, namespace, **kwargs):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # skip the base class itself
            key = namespace.get('name', name.lower())
            mcs._registry[key] = cls
            print(f'  Registered: {key!r} -> {cls.__qualname__}')
        return cls

class Serializer(metaclass=PluginMeta):
    def serialize(self, data) -> bytes: ...
    def deserialize(self, raw: bytes): ...

print('Defining serializers:')
class JSONSerializer(Serializer):
    name = 'json'
    def serialize(self, data) -> bytes:
        import json; return json.dumps(data).encode()

class PickleSerializer(Serializer):
    name = 'pickle'
    def serialize(self, data) -> bytes:
        import pickle; return pickle.dumps(data)

print(f'\nRegistry: {list(PluginMeta._registry.keys())}')
s = PluginMeta._registry['json']()
print(f'JSON serialized: {s.serialize({"x": 1})}')

print()
# ── __prepare__: ordered namespace trick ──────────────────────────────────
class OrderedMeta(type):
    """Track the order in which class attributes are defined."""
    @classmethod
    def __prepare__(mcs, name, bases):
        return {}  # dict is ordered since 3.7 (guaranteed)

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        cls._field_order = [k for k in namespace if not k.startswith('_')]
        return cls

class Record(metaclass=OrderedMeta):
    id    = None
    name  = None
    email = None
    age   = None

print(f'Field definition order: {Record._field_order}')

## 3. `__new__` vs `__init__`

| | `__new__(cls, ...)` | `__init__(self, ...)` |
|---|---|---|
| When | Before instance exists | After instance created |
| Returns | New instance | None |
| Use case | Immutable types, singletons, caching | Normal initialization |
| Called on | Class (`cls`) | Instance (`self`) |

In [ ]:
# ── Singleton via __new__ ──────────────────────────────────────────────────
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value):
        self.value = value

a = Singleton(1)
b = Singleton(2)
print(f'a is b: {a is b}')      # True — same instance
print(f'a.value: {a.value}')    # 2 — __init__ ran again on same object

print()
# ── Interned integers via __new__ (how CPython works) ──────────────────────
class InternedInt(int):
    _cache: dict[int, 'InternedInt'] = {}

    def __new__(cls, value):
        if value not in cls._cache:
            cls._cache[value] = super().__new__(cls, value)
        return cls._cache[value]

a, b = InternedInt(42), InternedInt(42)
c    = InternedInt(43)
print(f'InternedInt(42) is InternedInt(42): {a is b}')   # True
print(f'InternedInt(42) is InternedInt(43): {a is c}')   # False

## 4. Key takeaways

| Concept | Rule |
|---|---|
| Descriptor protocol | `__get__`/`__set__`/`__delete__` — the engine behind properties, methods, slots |
| Data descriptor | Has `__set__` — takes priority over instance `__dict__` |
| Non-data descriptor | Only `__get__` — instance dict wins (why you can shadow methods) |
| `__set_name__` | Called at class creation time with the attribute name |
| Metaclass | Class of a class — intercepts class creation; prefer `__init_subclass__` for simpler cases |
| `__new__` | Use for immutable types and singletons; `__init__` for everything else |

**Next:** notebook 06 — threading, the GIL, and `concurrent.futures`.